# 05 – Huấn luyện LUỒNG 2: MIL kiểu Sultani + head phân nhóm (UCF-Crime)

Sinh tự động bởi `realtime/tools/build_scene_train.py` – **không sửa tay**.

**Chuẩn bị – Add Input:** output của notebook **04_scene_features** (có `scene_seg32/`, `scene_meta.csv`,
`ucf_annotation.csv`). Settings → **GPU T4** · **Internet on** (chỉ để cài `onnx` và tải trọng số MobileNetV3 khi xuất
`scene_backbone.onnx`).

**Dữ liệu:** train bất thường = UCF (810 video) · train bình thường = RWF NonFight + RLVS NonViolence (~1600) ·
test = UCF bất thường (140) · Normal UCF (150, **ngoài miền**) · RWF / RLVS val (Fight / Violence chỉ đánh giá).
Lấy mẫu cân bằng theo `train_w` (theo phút). Không đệm 0: mỗi video đã là 32 đoạn cố định.

**Chạy:** `QUICK = True` thử → ổn thì `QUICK = False` (Save & Run All).

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("onnx") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript"])
    assert r.returncode == 0, "pip install loi -> bat Internet: Settings -> Internet on"
import onnx
print("onnx", onnx.__version__)

## 1. Code dùng chung với realtime

In [ ]:
%%writefile scene.py
"""LUỒNG 2 – nhìn ảnh màu cả cảnh (UCF-Crime). CLAUDE.md mục 12.

DÙNG CHUNG giữa notebook Kaggle (04 trích đặc trưng, 05 train) và realtime (detectors/scene.py):
- hằng số (2 FPS, 224 px, đoạn 16 frame), cách nhận LỚP từ đường dẫn video, ánh xạ lớp -> NHÓM;
- tiền xử lý 1 frame: BGR -> resize 224×224 (ép cả khung, không cắt) -> RGB uint8. Chuẩn hoá ImageNet nằm
  TRONG mô hình backbone (scene_backbone.onnx nhận uint8), nên realtime và lúc train không thể lệch nhau.
Chỉ phụ thuộc numpy + cv2.
"""
import re

import numpy as np

FPS = 2                     # lấy mẫu 2 frame / giây theo thời gian (shared/sampling.py)
SIZE = 224
SEGMENT = 16                # 16 frame mẫu = 8 s / đoạn
FEAT_DIM = 960              # MobileNetV3-Large: vector sau global average pooling
MAX_SEC = 15 * 60           # chỉ lấy 15 phút đầu mỗi video
VIDEO_EXT = {".mp4", ".avi", ".mpg", ".mpeg", ".mkv", ".mov", ".m4v", ".webm", ".wmv"}
SKIP_PATH = ("/anomaly/",)  # bản UCF mission-ai có thư mục /anomaly/ trùng -> bỏ
# Thư mục CẤM (so trên tên đã chuẩn hoá của từng thư mục trong đường dẫn): video biến đổi / gộp lại, có thể sinh từ
# video test -> rò rỉ; hoặc thư mục đặc trưng / mô hình có sẵn (không phải video gốc).
FORBIDDEN_EXACT = {"alldata": "All_Data"}
FORBIDDEN_SUB = {"augmented": "Augmented*", "fightinga": "FightingA*", "extracted": "Extracted*",
                 "perframe": "Per_Frame*", "modeltrained": "Model_Trained*"}
NO_SCAN_SUB = ("extracted", "perframe", "modeltrained")          # thư mục đặc trưng: không quét vào (rất nhiều file)
PREFERRED_DATASETS = ("anomalydetectiondatasetucf",)              # bản UCF đầy đủ (minhajuddinmeraj) – ưu tiên khi trùng
LOW_PRIORITY_DUP = ("real life violence situations",)   # bản sao trùng của RLVS -> chỉ giữ khi không có bản chính
NORMAL = "Normal"           # mọi thư mục / tên file có chữ "normal" -> lớp Normal
SOURCE_OF = {"Fight": "rwf", "NonFight": "rwf", "Violence": "rlvs", "NonViolence": "rlvs"}   # còn lại: ucf


def norm(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def class_index(groups, eval_only=None):
    """groups: {nhóm: [lớp, ...]} (config.yaml scene.groups); eval_only: {lớp: nhóm} chỉ dùng đánh giá.
    -> {tên chuẩn hoá: (Lớp, nhóm, dùng để train?)}"""
    idx = {}
    for g, classes in groups.items():
        for c in classes:
            idx[norm(c)] = (c, g, True)
    for c, g in (eval_only or {}).items():
        idx[norm(c)] = (c, g, False)
    return idx


def detect_class(path, index):
    """Lớp của video từ đường dẫn: thư mục cha gần nhất trùng tên lớp (không phân biệt hoa thường, bỏ dấu câu);
    thư mục có chữ "normal" -> Normal; nếu không thấy, thử tiền tố tên file (vd Abuse001_x264.mp4). None = bỏ."""
    parts = str(path).replace("\\", "/").split("/")
    for comp in reversed(parts[:-1]):
        n = norm(comp)
        if n in index:
            return index[n][0]
        if "normal" in n and norm(NORMAL) in index:
            return NORMAL
    name = parts[-1]
    m = re.match(r"[A-Za-z]+", name)
    if m and norm(m.group(0)) in index:
        return index[norm(m.group(0))][0]
    if "normal" in norm(name) and norm(NORMAL) in index:
        return NORMAL
    return None


def source_of(cls):
    return SOURCE_OF.get(cls, "ucf")


AUG_RE = re.compile(r"(?i)((?<=[0-9])[_-]?a|[_-](aug|flip|flipped|noise|blur|bright|dark|rot)[a-z0-9]*)$")


def key_of(name):
    """Khoá so khớp tên video / dòng danh sách: bỏ thư mục, đuôi, dấu, hoa thường, hậu tố x264 -> chữ và số."""
    base = str(name).strip().replace("\\", "/").split("/")[-1]
    stem = base.rsplit(".", 1)[0] if "." in base else base
    key = re.sub(r"[^a-z0-9]", "", stem.lower())
    return key[:-4] if key.endswith("x264") else key   # danh sach UCF co khi thieu hau to _x264


def is_augmented(name):
    """Bản augment (hậu tố A như Abuse001_x264A, hoặc _flip/_aug...): loại khỏi train VÀ test."""
    stem = str(name).replace("\\", "/").split("/")[-1].rsplit(".", 1)[0]
    return AUG_RE.search(stem) is not None


def base_of(name):
    """Khoá của video gốc ứng với một bản augment (bỏ hậu tố)."""
    stem = str(name).replace("\\", "/").split("/")[-1].rsplit(".", 1)[0]
    return key_of(AUG_RE.sub("", stem) or stem)


def skip_reason(rel_path):
    """Lý do loại video (None = giữ). rel_path: đường dẫn TƯƠNG ĐỐI trong /kaggle/input (tránh khớp nhầm thư mục gốc)."""
    p = "/" + str(rel_path).replace("\\", "/").lstrip("/")
    if any(s in p.lower() for s in SKIP_PATH):
        return "/anomaly/ (ban sao)"
    for comp in p.split("/")[:-1]:
        n = norm(comp)
        if n in FORBIDDEN_EXACT:
            return FORBIDDEN_EXACT[n]
        for sub, label in FORBIDDEN_SUB.items():
            if sub in n:
                return label
    return None


def skip_path(rel_path):
    return skip_reason(rel_path) is not None


def no_scan(dirname):
    """Thư mục đặc trưng / mô hình: không cần quét vào (rất nhiều file, không có video gốc)."""
    n = norm(dirname)
    return any(s in n for s in NO_SCAN_SUB)


def dup_rank(path):
    """Ưu tiên giữa các BẢN SAO cùng tên file (nhỏ hơn = giữ): 0 = bản chính, 1 = bản sao RLVS đã biết."""
    p = str(path).replace("\\", "/").lower()
    return int(any(s in p for s in LOW_PRIORITY_DUP))


def priority(rel_path, top_folder, index):
    """Khoá sắp xếp khi trùng tên file (nhỏ hơn = giữ):
    (dataset ưu tiên trước, thư mục chính trước thư mục lớp lẻ / Event_Recognition, bản chính trước bản sao).
    top_folder: thư mục cấp đầu ngay dưới gốc dataset (vd Anomaly-Videos-Part-1, Burglary)."""
    ds = 0 if any(d in norm(rel_path) for d in PREFERRED_DATASETS) else 1
    t = norm(top_folder)
    loose = int(t in index or "eventrecognition" in t)
    return ds, loose, dup_rank(rel_path)


def video_key(source, cls, path):
    """Khoá duy nhất (tên file .npy) – chống trùng theo (nguồn, lớp, tên file)."""
    stem = str(path).replace("\\", "/").split("/")[-1].rsplit(".", 1)[0]
    return re.sub(r"[^A-Za-z0-9_.-]", "_", f"{source}__{cls}__{stem}")


def prep(frame_bgr):
    """1 frame BGR (bất kỳ kích thước) -> RGB uint8 (224, 224, 3), ép cả khung (không cắt mất mép cảnh)."""
    import cv2
    small = cv2.resize(frame_bgr, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(small, cv2.COLOR_BGR2RGB)


def segments(n, segment=SEGMENT, stride=None):
    """Điểm bắt đầu các đoạn `segment` frame mẫu phủ n frame (đoạn cuối căn về cuối; n < segment -> [0])."""
    stride = stride or segment
    if n <= segment:
        return [0]
    starts = list(range(0, n - segment + 1, stride))
    if starts[-1] != n - segment:
        starts.append(n - segment)
    return starts


def parse_temporal_annotation(text):
    """Temporal_Anomaly_Annotation_for_Testing_Videos.txt -> {tên file: (lớp, [(frame_đầu, frame_cuối), ...])}.
    Mỗi dòng: <tên> <lớp> s1 e1 s2 e2 (frame ở FPS gốc, -1 = không có)."""
    out = {}
    for line in text.splitlines():
        p = line.split()
        if len(p) < 6:
            continue
        nums = [int(float(x)) for x in p[2:6]]
        spans = [(s, e) for s, e in (nums[0:2], nums[2:4]) if s >= 0 and e >= 0]
        out[p[0]] = (p[1], spans)
    return out


def parse_test_list(text):
    """Anomaly_Train.txt / Anomaly_Test.txt / Normal_Train.txt (vd "Abuse/Abuse028_x264.mp4") -> tập tên file
    (chữ thường, chỉ lấy dòng có đuôi video)."""
    out = set()
    for ln in text.splitlines():
        name = ln.strip().replace("\\", "/").split("/")[-1].split()[0] if ln.strip() else ""
        if name and "." + name.rsplit(".", 1)[-1].lower() in VIDEO_EXT:
            out.add(name.lower())
    return out


def frame_labels(times, fps_src, spans):
    """Nhãn bất thường (0/1) cho các frame mẫu tại thời điểm `times` (giây) theo khoảng chú thích (frame gốc)."""
    t = np.asarray(times, float)
    y = np.zeros(len(t), np.int8)
    for s, e in spans:
        y[(t >= s / fps_src) & (t <= e / fps_src)] = 1
    return y

In [ ]:
%%writefile scene_groups.json
{
 "groups": {
  "BẠO LỰC": [
   "Fighting",
   "Assault",
   "Abuse",
   "Arrest",
   "Shooting"
  ],
  "CƯỚP": [
   "Robbery"
  ],
  "CHÁY NỔ": [
   "Arson",
   "Explosion"
  ],
  "ĐỘT NHẬP/TRỘM": [
   "Burglary",
   "Stealing",
   "Shoplifting",
   "Vandalism"
  ],
  "TAI NẠN": [
   "RoadAccidents"
  ],
  "BÌNH THƯỜNG": [
   "Normal",
   "NonFight",
   "NonViolence"
  ]
 },
 "normal_group": "BÌNH THƯỜNG",
 "eval_only": {
  "Fight": "BẠO LỰC",
  "Violence": "BẠO LỰC"
 }
}

## 2. Cấu hình

In [ ]:
import os
QUICK = True              # True: thu nhanh. Ok roi doi False de chay day du.
QUICK_EPOCHS, QUICK_STEPS = 2, 20
EPOCHS, STEPS = 20, 200
BAG = 8                   # moi buoc: BAG video bat thuong + BAG video binh thuong
LR, WD = 1e-3, 1e-4
SEED = 42
EXPORT_BACKBONE = True    # xuat scene_backbone.onnx (can Internet de tai trong so ImageNet)
INPUT = os.environ.get("HTTM_INPUT", "/kaggle/input")
WORK = os.environ.get("HTTM_WORK", "/kaggle/working")
os.makedirs(WORK, exist_ok=True)

import os, sys, glob, json, time, importlib, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from sklearn.metrics import roc_auc_score, f1_score
sys.path.insert(0, os.getcwd())
import scene as S
importlib.reload(S)
CFG = json.load(open("scene_groups.json", encoding="utf-8"))
NORMAL = CFG["normal_group"]
ABN_GROUPS = [g for g in CFG["groups"] if g != NORMAL]
print("5 nhom bat thuong:", ABN_GROUPS)
torch.manual_seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)
print(f"QUICK = {QUICK} | MAX_SEC (15 phut khi chay day du) = {S.MAX_SEC} s")

## 3. Dữ liệu từ notebook 04

In [ ]:
seg_dirs = sorted(d for d in glob.glob(os.path.join(INPUT, "**", "scene_seg32*"), recursive=True) if os.path.isdir(d))
assert seg_dirs, "Khong thay scene_seg32/ -> Add Input output cua notebook 04_scene_features"
full = [d for d in seg_dirs if os.path.basename(d) == "scene_seg32"]
SEG = (full or seg_dirs)[0]
print("Doan 32:", SEG, "| cac thu muc tim thay:", seg_dirs)
META = os.path.join(os.path.dirname(SEG), "scene_meta.csv")
if not os.path.exists(META):
    META = glob.glob(os.path.join(INPUT, "**", "scene_meta.csv"), recursive=True)[0]
ANN_F = (glob.glob(os.path.join(INPUT, "**", "ucf_annotation.csv"), recursive=True) or [None])[0]
meta = pd.read_csv(META)
meta = meta[meta.key.map(lambda k: os.path.exists(os.path.join(SEG, k + ".npy")))].reset_index(drop=True)
meta["ab"] = meta.group != NORMAL
meta["gi"] = meta.group.map({g: i for i, g in enumerate(ABN_GROUPS)}).fillna(-1).astype(int)
meta["dur_used"] = np.minimum(meta.n_read / meta.fps, S.MAX_SEC)
print(f"{len(meta)} video co doan 32 | meta: {META}")
print("Thoi luong trung binh (giay) theo nhom:", meta.groupby(["ab", "split"]).dur_used.mean().round(1).to_dict())

In [ ]:
TR_AB = meta.index[(meta.split == "train") & meta.ab & meta.train_use].values
TR_NO = meta.index[(meta.split == "train") & ~meta.ab & meta.train_use].values
TE_UCF_AB = meta.index[(meta.split == "test") & meta.ab & (meta.source == "ucf")].values
TE_UCF_NO = meta.index[(meta.split == "test") & ~meta.ab & (meta.source == "ucf")].values
TE_RWF = meta.index[(meta.split == "test") & (meta.source == "rwf")].values
TE_RLVS = meta.index[(meta.split == "test") & (meta.source == "rlvs")].values
print("== So video")
print(pd.Series({"train bat thuong (UCF)": len(TR_AB), "train binh thuong (RWF NonFight + RLVS NonViolence)": len(TR_NO),
                 "test bat thuong UCF": len(TE_UCF_AB), "test Normal UCF (ngoai mien)": len(TE_UCF_NO),
                 "test RWF (Fight + NonFight)": len(TE_RWF), "test RLVS (Violence + NonViolence)": len(TE_RLVS)}))
print("Ky vong: 810 | 1600 | 140 | 150 | - | -   (QUICK chi co mot phan)")
print("Nhan nhom tren tap train bat thuong:", meta.loc[TR_AB].group.value_counts().to_dict())

## 4. Chuẩn hoá đặc trưng (thống kê trên các đoạn train)

In [ ]:
XS = np.stack([np.load(os.path.join(SEG, k + ".npy")) for k in meta.key])        # (N, 32, 960) float16
print("XS", XS.shape, XS.dtype, f"{XS.nbytes / 1e9:.2f} GB")
tr_idx = np.r_[TR_AB, TR_NO]
samp = XS[tr_idx].reshape(-1, 960)
samp = samp[rng.choice(len(samp), min(len(samp), 20000), replace=False)].astype(np.float32)
MU, SD = samp.mean(0), np.maximum(samp.std(0), 1e-3)

def norm(x):
    return (x.astype(np.float32) - MU) / SD

def wts(idx):
    w = np.maximum(meta.train_w.values[idx].astype(float), 1e-9)
    return w / w.sum()                                       # lay mau ti le train_w (can bang theo phut)

## 5. Mô hình và huấn luyện (MIL kiểu Sultani)
Điểm đoạn `s_i` (32 đoạn / video). **Ranking loss:** `max(0, 1 − max_i s_abn + max_j s_nor)` + làm mượt theo thời gian
+ thưa (`s_abn` nhỏ). **Head nhóm:** logit 5 nhóm cấp video, **chỉ học trên video bất thường**.

In [ ]:
class SceneHead(nn.Module):
    def __init__(self, d=960, h=128, p=0.3, n_groups=len(ABN_GROUPS)):
        super().__init__()
        self.enc = nn.Sequential(nn.Dropout(p), nn.Linear(d, h), nn.ReLU(), nn.Dropout(p))
        self.score = nn.Linear(h, 1)
        self.group = nn.Linear(h, n_groups)

    def forward(self, x):                                    # (B, 32, 960) -> diem doan (B, 32), logit nhom (B, 5)
        h = self.enc(x)
        return self.score(h).squeeze(-1), self.group(h.mean(1))

model = SceneHead().to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
n_ep, n_st = (QUICK_EPOCHS, QUICK_STEPS) if QUICK else (EPOCHS, STEPS)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_ep * n_st)
ce = nn.CrossEntropyLoss()
t0 = time.time()
for ep in range(n_ep):
    model.train()
    tot = []
    for _ in range(n_st):
        a = rng.choice(TR_AB, BAG, p=wts(TR_AB))
        n = rng.choice(TR_NO, BAG, p=wts(TR_NO))
        sa, ga = model(torch.from_numpy(norm(XS[a])).to(DEVICE))
        sn, _ = model(torch.from_numpy(norm(XS[n])).to(DEVICE))
        hinge = torch.relu(1 - sa.max(1).values + sn.max(1).values).mean()
        smooth = ((sa[:, 1:] - sa[:, :-1]) ** 2).sum(1).mean()
        sparse = sa.abs().sum(1).mean()
        gi = torch.tensor(meta.gi.values[a], device=DEVICE)
        loss = hinge + 8e-5 * smooth + 8e-5 * sparse + ce(ga, gi)     # ga: chi bag bat thuong
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        tot.append([loss.item(), hinge.item()])
    m = np.mean(tot, 0)
    print(f"epoch {ep + 1}/{n_ep} | loss {m[0]:.4f} | ranking {m[1]:.4f} | {time.time() - t0:.0f} s")

## 6. Đánh giá

In [ ]:
@torch.no_grad()
def scores(idx):
    model.eval()
    S_, G_ = [], []
    for c in range(0, len(idx), 128):
        s, g = model(torch.from_numpy(norm(XS[idx[c:c + 128]])).to(DEVICE))
        S_.append(s.cpu().numpy()); G_.append(g.cpu().numpy())
    return np.concatenate(S_), np.concatenate(G_)

def auc(y, s):
    return float(roc_auc_score(y, s)) if len(set(y)) == 2 else float("nan")

s_tr_no = scores(TR_NO)[0].max(1)
THR = float(np.percentile(s_tr_no, 95))          # nguong: 95% diem video binh thuong train
print(f"Nguong (95% diem binh thuong train) = {THR:.3f}")

rows = []
for name, ab_idx, no_idx in [("UCF test (ngoai mien Normal)", TE_UCF_AB, TE_UCF_NO),
                             ("RWF test (Fight vs NonFight)", TE_RWF[meta.ab.values[TE_RWF]],
                              TE_RWF[~meta.ab.values[TE_RWF]]),
                             ("RLVS test (Violence vs NonViolence)", TE_RLVS[meta.ab.values[TE_RLVS]],
                              TE_RLVS[~meta.ab.values[TE_RLVS]])]:
    idx = np.r_[ab_idx, no_idx]
    if len(idx) == 0:
        continue
    y = np.r_[np.ones(len(ab_idx)), np.zeros(len(no_idx))].astype(int)
    sc = scores(idx)[0].max(1)
    pred = (sc > THR).astype(int)
    rows.append(dict(tap=name, n_bat_thuong=len(ab_idx), n_binh_thuong=len(no_idx), AUC=auc(y, sc),
                     F1=f1_score(y, pred, zero_division=0),
                     bao_nham_tren_binh_thuong=float(pred[y == 0].mean()) if (y == 0).any() else float("nan")))
res = pd.DataFrame(rows)
print("== AUC / F1 theo tung tap test (nguong 95% binh thuong train)")
display(res.round(3))
if len(TE_UCF_NO):
    fp = float((scores(TE_UCF_NO)[0].max(1) > THR).mean())
    print(f"TY LE BAO NHAM tren {len(TE_UCF_NO)} Normal UCF (ngoai mien): {fp:.1%}")

# AUC muc doan (~ muc frame) tren UCF test theo chu thich thoi gian
spans = {}
if ANN_F:
    ann = pd.read_csv(ANN_F)
    for _, r in ann.iterrows():
        sp = [(int(r[a]), int(r[b])) for a, b in (("s1", "e1"), ("s2", "e2")) if r[a] >= 0 and r[b] >= 0]
        spans[str(r.fname).lower()] = sp
idx_fr = np.r_[TE_UCF_AB, TE_UCF_NO]
fr_y, fr_s = [], []
s_all = scores(idx_fr)[0]
for j, i in enumerate(idx_fr):
    t = (np.arange(32) + 0.5) / 32 * meta.dur_used.values[i]
    sp = spans.get(str(meta.fname.values[i]).lower(), [])
    fr_y.append(S.frame_labels(t, float(meta.fps.values[i]), sp) if sp else np.zeros(32, np.int8))
    fr_s.append(s_all[j])
fr_y, fr_s = np.concatenate(fr_y), np.concatenate(fr_s)
print(f"AUC MUC DOAN (~frame) tren UCF test: {auc(fr_y, fr_s):.3f} | doan duong: {int(fr_y.sum())} / {len(fr_y)}")

# nhom: dung tren video bat thuong test
gi_t = meta.gi.values[TE_UCF_AB]
_, g_logits = scores(TE_UCF_AB)
ok = gi_t >= 0
acc = float((g_logits.argmax(1)[ok] == gi_t[ok]).mean()) if ok.any() else float("nan")
print(f"DO CHINH XAC PHAN NHOM (video bat thuong UCF test, {ok.sum()} video): {acc:.1%}")
print(pd.crosstab(pd.Series(np.array(ABN_GROUPS)[gi_t[ok]], name="that"),
                  pd.Series(np.array(ABN_GROUPS)[g_logits.argmax(1)[ok]], name="du doan")))

# kiem tra loi tat do dai: diem TB theo nhom do dai clip, va AUC cua chinh do dai
lens = []
for name, idx, lab in [("UCF test", np.r_[TE_UCF_AB, TE_UCF_NO], np.r_[np.ones(len(TE_UCF_AB)), np.zeros(len(TE_UCF_NO))]),
                       ("RWF test", TE_RWF, meta.ab.values[TE_RWF].astype(int)),
                       ("RLVS test", TE_RLVS, meta.ab.values[TE_RLVS].astype(int))]:
    if len(idx) == 0:
        continue
    sc = scores(idx)[0].max(1)
    print(f"{name}: AUC chi dung DO DAI = {auc(lab, meta.dur_used.values[idx]):.3f}  (AUC mo hinh = {auc(lab, sc):.3f})")
    lens.append(pd.DataFrame(dict(tap=name, nhan=lab, do_dai=pd.cut(meta.dur_used.values[idx],
                                                                    [0, 10, 60, 300, S.MAX_SEC]),
                                  diem=sc)))
if lens:
    print("== Diem trung binh theo do dai clip (kiem tra loi tat)")
    display(pd.concat(lens).groupby(["tap", "nhan", "do_dai"], observed=True).diem.agg(["count", "mean"]).round(3))

In [ ]:
class SceneBackbone(nn.Module):
    """MobileNetV3-Large pretrained ImageNet, ĐÓNG BĂNG. Vào: (B, 224, 224, 3) uint8 RGB (đầu ra scene.prep).
    Ra: (B, 960) – vector sau global average pooling. Chuẩn hoá ImageNet nằm TRONG mô hình."""
    def __init__(self, weights):
        super().__init__()
        m = torchvision.models.mobilenet_v3_large(weights=weights)
        self.features, self.pool = m.features, m.avgpool
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1) * 255)
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1) * 255)

    def forward(self, x):
        x = (x.permute(0, 3, 1, 2).float() - self.mean) / self.std
        return self.pool(self.features(x)).flatten(1)

## 7. Xuất `scene_head.onnx`, `scene_backbone.onnx`, `scene_config.json`

In [ ]:
model.eval().cpu()
torch.save({"model": model.state_dict(), "mu": MU, "sd": SD}, os.path.join(WORK, "scene_head.pt"))
head_onnx = os.path.join(WORK, "scene_head.onnx")
torch.onnx.export(model, torch.zeros(2, 32, 960), head_onnx, input_names=["x"],
                  output_names=["seg_score", "group_logits"], opset_version=17,
                  dynamic_axes={"x": {0: "batch"}, "seg_score": {0: "batch"}, "group_logits": {0: "batch"}},
                  dynamo=False)
import onnxruntime as ort
sess = ort.InferenceSession(head_onnx, providers=["CPUExecutionProvider"])
xx = norm(XS[TE_UCF_AB[:4]] if len(TE_UCF_AB) else XS[:4])
ref_s, ref_g = model(torch.from_numpy(xx))
out_s, out_g = sess.run(None, {"x": xx})
print(f"ONNX head vs PyTorch: lech {np.abs(out_s - ref_s.detach().numpy()).max():.2e} / {np.abs(out_g - ref_g.detach().numpy()).max():.2e}")

if EXPORT_BACKBONE:
    try:
        bb = SceneBackbone(torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V2).eval()
    except Exception as e:
        print("!! Khong tai duoc trong so MobileNetV3 (can Internet):", e)
        bb = None
    if bb is not None:
        torch.onnx.export(bb, torch.zeros(2, S.SIZE, S.SIZE, 3, dtype=torch.uint8),
                          os.path.join(WORK, "scene_backbone.onnx"), input_names=["frame_rgb_uint8"],
                          output_names=["feat960"], opset_version=17,
                          dynamic_axes={"frame_rgb_uint8": {0: "batch"}, "feat960": {0: "batch"}}, dynamo=False)
        print("scene_backbone.onnx da xuat (uint8 RGB 224x224 -> 960)")

cfg_out = {"fps": S.FPS, "size": S.SIZE, "feat_dim": S.FEAT_DIM, "n_seg": 32, "segment_note":
           "train tren 32 doan cua video; realtime phai tao 32 doan tu chuoi frame cung cach (xem scene.py)",
           "groups_abnormal": ABN_GROUPS, "normal_group": NORMAL, "threshold": THR,
           "mean": MU.tolist(), "std": SD.tolist(), "trained_on": "UCF train + RWF NonFight + RLVS NonViolence",
           "quick": QUICK, "results": res.to_dict("records")}
with open(os.path.join(WORK, "scene_config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg_out, f, ensure_ascii=False, indent=1, default=float)
for fn in sorted(os.listdir(WORK)):
    if fn.startswith("scene_"):
        print(f"{fn:24s} {os.path.getsize(os.path.join(WORK, fn)) / 1e6:8.2f} MB")